# Prompt Templates & Prompt Engineering with LangChain

This notebook covers everything a beginner needs to know about **prompts**:

1. **What is a prompt?** — The text you send to an AI model
2. **Prompt Templates** — Reusable prompts with variables (like fill-in-the-blank)
3. **Chat Prompt Templates** — Templates for chat-style models (system + human messages)
4. **Prompt Engineering Techniques** — Zero-shot, Few-shot, Chain-of-Thought, Role-playing, etc.
5. **LangChain Hub** — Download and share ready-made prompts
6. **HuggingFace Prompt Resources** — Open-source prompts and datasets

---

## Setup

In [1]:
!pip install langchain langchain-openai langchain-huggingface langchainhub python-dotenv -q

In [17]:
import os 
os.chdir(r"D:\AI_Products_Dev\Learning\LangChain_Models")
from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

print("Ready!")

Ready!


---

# Part 1 — Prompt Templates

## What is a Prompt Template?

A **Prompt Template** is like a fill-in-the-blank sentence. You write the prompt once with **placeholders**, then fill in the blanks later.

**Without template** (hardcoded):
```
"Translate 'hello' to French"
"Translate 'goodbye' to Spanish"
"Translate 'thanks' to Japanese"
```

**With template** (reusable):
```
"Translate '{word}' to {language}"
```

Much cleaner!

## 1.1 — PromptTemplate (simple string prompts)

In [19]:
from langchain_core.prompts import PromptTemplate

# Create a template with placeholders in {curly braces}
template = PromptTemplate.from_template(
    "Translate the word '{word}' to {language}. Just give the translation, nothing else."
)

# Fill in the blanks
prompt = template.format(word="hello", language="French")
print("Generated prompt:", prompt)

Generated prompt: Translate the word 'hello' to French. Just give the translation, nothing else.


In [21]:
# Now send it to the AI model
response = llm.invoke(prompt)
print(response.content)

bonjour


In [ ]:
# Reuse the SAME template with different values
for word, lang in [("goodbye", "Spanish"), ("thank you", "Japanese"), ("water", "Hindi")]:
    prompt = template.format(word=word, language=lang)
    response = llm.invoke(prompt)
    print(f"{word} → {lang}: {response.content}")

## 1.2 — PromptTemplate with multiple variables

In [ ]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate.from_template(
    "Explain {topic} to a {audience} in {sentences} sentences."
)

prompt = template.format(topic="machine learning", audience="5-year-old", sentences="3")
print("Prompt:", prompt)
print()

response = llm.invoke(prompt)
print("Answer:", response.content)

In [ ]:
# Same template, different audience
prompt = template.format(topic="machine learning", audience="college student", sentences="3")

response = llm.invoke(prompt)
print("Answer:", response.content)

## 1.3 — Using Chains (Template + Model together)

Instead of calling `.format()` then `.invoke()` separately, you can **chain** them with the `|` pipe operator.

In [ ]:
# Reuse with different input
response = chain.invoke({"topic": "the moon"})
print(response.content)

---

# Part 2 — Chat Prompt Templates

Chat models (like GPT-4, Claude) expect **messages** not plain strings. There are 3 types:

| Message Type | Who is speaking? | Purpose |
|-------------|-----------------|----------|
| `system` | You (hidden from user) | Set the AI's behavior / personality |
| `human` | The user | The actual question |
| `ai` | The AI | Previous AI responses (for context) |

## 2.1 — ChatPromptTemplate basics

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}. Always respond in {style} style."),
    ("human", "{question}"),
])

# Fill in all the variables
messages = chat_template.format_messages(
    role="pirate captain",
    style="pirate",
    question="What is Python programming?"
)

# See what messages look like
for msg in messages:
    print(f"{msg.type}: {msg.content}")

In [ ]:
# Send to AI
response = llm.invoke(messages)
print(response.content)

In [ ]:
# Same template, different role
messages = chat_template.format_messages(
    role="Shakespearean poet",
    style="poetic",
    question="What is Python programming?"
)

response = llm.invoke(messages)
print(response.content)

## 2.2 — ChatPromptTemplate as a chain

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are an expert {subject} teacher. Explain things simply."),
    ("human", "{question}"),
])

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

chain = chat_template | llm

response = chain.invoke({"subject": "physics", "question": "Why is the sky blue?"})
print(response.content)

---

# Part 3 — Prompt Engineering Techniques

**Prompt Engineering** = the art of writing better prompts to get better answers.

Here are the most important techniques:

## 3.1 — Zero-Shot Prompting

Just ask directly — no examples given. The model uses its own knowledge.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

zero_shot = ChatPromptTemplate.from_messages([
    ("system", "Classify the sentiment of the text as Positive, Negative, or Neutral."),
    ("human", "{text}"),
])

chain = zero_shot | llm

response = chain.invoke({"text": "I absolutely love this new phone! Best purchase ever!"})
print("Result:", response.content)

In [ ]:
response = chain.invoke({"text": "The delivery was late and the box was damaged."})
print("Result:", response.content)

## 3.2 — Few-Shot Prompting

Give the model a few **examples** first, then ask it to do the same. This teaches the model the pattern you want.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

few_shot = ChatPromptTemplate.from_messages([
    ("system", "You classify customer reviews into categories: Product, Delivery, Service, Price."),
    ("human", "The screen quality is amazing!"),
    ("ai", "Product"),
    ("human", "It arrived 2 days late."),
    ("ai", "Delivery"),
    ("human", "The support team was very rude."),
    ("ai", "Service"),
    ("human", "Way too expensive for what you get."),
    ("ai", "Price"),
    ("human", "{review}"),
])

chain = few_shot | llm

# Now test with new reviews
test_reviews = [
    "The battery lasts all day!",
    "Package was left in the rain.",
    "Customer care resolved my issue quickly.",
    "Great value for money!",
]

for review in test_reviews:
    response = chain.invoke({"review": review})
    print(f"\"{review}\" → {response.content}")

## 3.3 — Few-Shot with FewShotPromptTemplate

LangChain has a built-in way to manage few-shot examples cleanly.

In [ ]:
chain = few_shot_template | llm

for word in ["hot", "heavy", "loud", "old"]:
    response = chain.invoke({"input": word})
    print(f"{word} → {response.content.strip()}")

## 3.4 — Chain-of-Thought (CoT) Prompting

Ask the model to **think step by step**. This dramatically improves reasoning and math.

In [ ]:
# WITHOUT chain-of-thought
simple = ChatPromptTemplate.from_messages([
    ("system", "Answer the question."),
    ("human", "{question}"),
])

chain = simple | llm

question = "A store has 45 apples. They sell 12 in the morning, get a delivery of 30, then sell 18 in the evening. How many apples are left?"

response = chain.invoke({"question": question})
print("Without CoT:")
print(response.content)

In [ ]:
# WITH chain-of-thought
cot = ChatPromptTemplate.from_messages([
    ("system", "Answer the question. Think step by step, show your work, then give the final answer."),
    ("human", "{question}"),
])

chain = cot | llm

response = chain.invoke({"question": question})
print("With CoT:")
print(response.content)

## 3.5 — Role-Based Prompting

Give the AI a specific **role/persona** — it changes how it responds.

In [ ]:
role_template = ChatPromptTemplate.from_messages([
    ("system", 
     "You are {role}. "
     "Respond from this perspective. "
     "Use language and examples appropriate for your role."),
    ("human", "{question}"),
])

question = "Why should I learn Python?"

roles = [
    "a senior software engineer at Google",
    "a data scientist at a hospital",
    "a high school computer science teacher",
]

chain = role_template | llm

for role in roles:
    response = chain.invoke({"role": role, "question": question})
    print(f"\n--- {role.upper()} ---")
    print(response.content)

## 3.6 — Output Format Control

Tell the model exactly **how** you want the answer formatted.

In [ ]:
# Get response as JSON
json_template = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful assistant. "
     "Always respond in valid JSON format with these keys: "
     "answer, confidence (low/medium/high), source."),
    ("human", "{question}"),
])

chain = json_template | llm

response = chain.invoke({"question": "What is the capital of France?"})
print(response.content)

In [ ]:
# Get response as a table
table_template = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful assistant. "
     "Always respond in a markdown table format."),
    ("human", "{question}"),
])

chain = table_template | llm

response = chain.invoke({"question": "Compare Python, JavaScript, and Java in terms of use case, difficulty, and popularity."})
print(response.content)

## 3.7 — Prompt Chaining (multi-step)

Use the output of one prompt as input to another.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Step 1: Generate a topic summary
step1 = ChatPromptTemplate.from_messages([
    ("system", "Summarize the topic in 2 sentences."),
    ("human", "{topic}"),
])

# Step 2: Generate quiz questions from the summary
step2 = ChatPromptTemplate.from_messages([
    ("system", "Based on this summary, create 3 multiple-choice quiz questions."),
    ("human", "{summary}"),
])

# Chain: step1 → get text → step2 → get text
chain1 = step1 | llm | StrOutputParser()
chain2 = step2 | llm | StrOutputParser()

# Run step 1
summary = chain1.invoke({"topic": "Photosynthesis"})
print("Summary:", summary)
print()

# Run step 2 with the output of step 1
quiz = chain2.invoke({"summary": summary})
print("Quiz:")
print(quiz)

## 3.8 — Delimiter / Structured Input

Use delimiters to clearly separate different parts of the prompt. Prevents prompt injection.

In [ ]:
delimiter_template = ChatPromptTemplate.from_messages([
    ("system",
     "You will be given a text delimited by triple backticks. "
     "Summarize it in exactly one sentence."),
    ("human", "```{text}```"),
])

chain = delimiter_template | llm

long_text = (
    "Artificial intelligence has transformed many industries including healthcare, "
    "finance, and education. In healthcare, AI helps doctors diagnose diseases earlier. "
    "In finance, it detects fraudulent transactions. In education, it personalizes "
    "learning paths for each student. However, there are also concerns about job "
    "displacement and privacy."
)

response = chain.invoke({"text": long_text})
print(response.content)

---

# Part 4 — LangChain Hub

**LangChain Hub** is a collection of ready-made prompts that you can download and use instantly.

- Browse prompts at: https://smith.langchain.com/hub
- Anyone can share their prompts
- You can pull prompts by their ID

Think of it like **npm for prompts** — someone else already wrote the perfect prompt for your task!

## 4.1 — Pull a prompt from LangChain Hub

In [ ]:
from langchain import hub

# Pull a popular RAG (Retrieval Augmented Generation) prompt
rag_prompt = hub.pull("rlm/rag-prompt")

# See what the prompt looks like
print("Type:", type(rag_prompt).__name__)
print()
print(rag_prompt.pretty_print())

In [ ]:
# Use the pulled prompt
chain = rag_prompt | llm

response = chain.invoke({
    "context": "LangChain is a framework for building LLM applications. It was created by Harrison Chase in 2022.",
    "question": "Who created LangChain?"
})

print(response.content)

## 4.2 — Explore different Hub prompts

In [ ]:
# Pull a summarization prompt
summarize_prompt = hub.pull("rlm/map-prompt")

print("Summarization prompt:")
print(summarize_prompt.pretty_print())

In [ ]:
chain = summarize_prompt | llm

text = (
    "Python is a high-level programming language created by Guido van Rossum in 1991. "
    "It emphasizes code readability with its use of significant indentation. Python supports "
    "multiple programming paradigms including procedural, object-oriented, and functional. "
    "It has a large standard library and active community, making it popular for web development, "
    "data science, AI, and automation."
)

response = chain.invoke({"docs": text})
print(response.content)

## 4.3 — Popular Hub Prompts to try

Here are some useful prompt IDs you can pull:

| Prompt ID | What it does |
|-----------|-------------|
| `rlm/rag-prompt` | RAG — answer questions from context |
| `rlm/map-prompt` | Summarize documents |
| `hwchase17/react` | ReAct agent — reasoning + tool use |
| `langchain-ai/sql-query-gen` | Generate SQL from natural language |

Browse all at: https://smith.langchain.com/hub

---

# Part 5 — HuggingFace Prompt Resources

HuggingFace offers:
1. **Open-source models** with their own prompt formats
2. **Datasets of prompts** for training and testing
3. **Community templates** for various tasks

Each open-source model has its own **chat template** (how it expects messages to be formatted).

## 5.1 — Using prompt templates with HuggingFace models

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

endpoint = HuggingFaceEndpoint(
    repo_id="HuggingFaceH4/zephyr-7b-beta",
    task="text-generation",
    max_new_tokens=256,
)

hf_llm = ChatHuggingFace(llm=endpoint)

# Same LangChain prompt template works with HuggingFace models!
template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful coding tutor. Keep answers short and simple."),
    ("human", "{question}"),
])

chain = template | hf_llm

response = chain.invoke({"question": "What is a for loop in Python?"})
print(response.content)

## 5.2 — Few-shot prompting with HuggingFace models

In [ ]:
few_shot_hf = ChatPromptTemplate.from_messages([
    ("system", "You convert English to SQL queries."),
    ("human", "Get all users"),
    ("ai", "SELECT * FROM users;"),
    ("human", "Get names of users older than 25"),
    ("ai", "SELECT name FROM users WHERE age > 25;"),
    ("human", "{question}"),
])

chain = few_shot_hf | hf_llm

response = chain.invoke({"question": "Count users from India"})
print(response.content)

## 5.3 — HuggingFace prompt datasets

HuggingFace hosts prompt datasets you can use for learning, testing, or fine-tuning.

| Dataset | What it contains | Link |
|---------|-----------------|------|
| `fka/awesome-chatgpt-prompts` | 170+ creative system prompts | [Link](https://huggingface.co/datasets/fka/awesome-chatgpt-prompts) |
| `Anthropic/hh-rlhf` | Human-rated helpful/harmful prompts | [Link](https://huggingface.co/datasets/Anthropic/hh-rlhf) |
| `tatsu-lab/alpaca` | 52K instruction-following prompts | [Link](https://huggingface.co/datasets/tatsu-lab/alpaca) |
| `Open-Orca/OpenOrca` | Millions of diverse prompts | [Link](https://huggingface.co/datasets/Open-Orca/OpenOrca) |

In [ ]:
!pip install datasets -q

In [ ]:
from datasets import load_dataset

# Load the awesome-chatgpt-prompts dataset
ds = load_dataset("fka/awesome-chatgpt-prompts", split="train")

# Show first 5 prompts
for i in range(5):
    print(f"\n--- {ds[i]['act']} ---")
    print(ds[i]['prompt'][:200], "...")

In [ ]:
# Use one of the HuggingFace prompts with LangChain
# Pick the "Linux Terminal" prompt
linux_prompt = ds[0]

print(f"Role: {linux_prompt['act']}")
print(f"Prompt: {linux_prompt['prompt'][:150]}...")
print()

template = ChatPromptTemplate.from_messages([
    ("system", linux_prompt["prompt"]),
    ("human", "{command}"),
])

chain = template | llm

response = chain.invoke({"command": "ls -la"})
print(response.content)

---

# Part 6 — Prompt Engineering Best Practices

| Tip | Example |
|-----|--------|
| **Be specific** | "Summarize in 3 bullet points" instead of "Summarize" |
| **Set the role** | "You are a senior Python developer" |
| **Give examples** | Show 2-3 input/output pairs (few-shot) |
| **Ask for step-by-step** | "Think step by step" (chain-of-thought) |
| **Control the format** | "Respond in JSON / table / bullet points" |
| **Use delimiters** | Wrap inputs in \`\`\` or \"\"\" to separate data from instructions |
| **Set constraints** | "Use max 50 words" or "Don't use technical jargon" |
| **Iterate** | Try → Check → Improve → Repeat |

## Quick reference — all prompt template types

```python
# 1. Simple string template
from langchain_core.prompts import PromptTemplate
template = PromptTemplate.from_template("Tell me about {topic}")

# 2. Chat template (system + human messages)
from langchain_core.prompts import ChatPromptTemplate
template = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}"),
    ("human", "{question}"),
])

# 3. Few-shot template (with examples)
from langchain_core.prompts import FewShotPromptTemplate

# 4. Messages placeholder (for conversation history)
from langchain_core.prompts import MessagesPlaceholder

# 5. Pull from LangChain Hub
from langchain import hub
template = hub.pull("rlm/rag-prompt")
```

**All of them work with any model** — OpenAI, Claude, Gemini, HuggingFace, Groq, etc.!